In [0]:
%run ../config/config

In [0]:
# Imports
import sys
import time
import os

sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.month_delta import month_delta

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

from utils.data_preparation.config import get_dataprep_config
from utils.data_preparation.fuente_hm_atraso_intra import FuenteHmAtrasoIntra
from utils.data_preparation.universo_implementacion import UniversoImplementacion

In [0]:
# Logger load preparation data
logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': '03_load_preparation_data'
    }
)

In [0]:
RUN_HISTORICAL = first_run
if RUN_HISTORICAL and num_meses_historicos > 0:
    meses_historicos_lista = []
    mes_tmp = mes_vta
    for i in range(num_meses_historicos):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [mes_vta]
    logger.info(f"MODO HISTÓRICO: {len(meses_a_procesar)} meses: {meses_a_procesar}")
else:
    meses_a_procesar = [mes_vta]
    logger.info(f"Procesando únicamente el mes actual: {mes_vta}")

spark = SparkSession.builder.getOrCreate()
resultados = []

for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("=" * 80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("=" * 80)

    try:
        logger.log_stage_start('load_preparation_data', mes_vta=mes_actual_proceso, environment=env)
        start_time = time.time()

        # --- PREPARACIÓN DE DATOS REFACTORIZADA ---
        logger.info(f"Obteniendo configuración DataPrep para periodo={mes_actual_proceso}")
        dp_config = get_dataprep_config(periodo=str(mes_actual_proceso))

        logger.info(f"Ejecutando UniversoImplementacion para periodo={mes_actual_proceso}")
        universo_imp = UniversoImplementacion(
            spark=spark,
            codmes_ini=dp_config["codmes_ini"],
            codmes_fin=dp_config["codmes_fin"],
            src_catalog=dp_config["src_catalog"],
            src_schema_portafolio=dp_config["src_schema_portafolio"],
            src_table_portafolio=dp_config["src_table_portafolio"],
            # Utilizar configuración global (config.ipynb) para el destino troncal lógico
            sink_catalog=catalog_name,
            sink_schema=schema_name,
            sink_table_portafolio_troncal=BASE_NAME_TABLE_MASTER_ENV,
            sink_table_hm_atraso=dp_config["sink_table_hm_atraso"]
        )
        universo_imp.execute()
        # --------------------------------
        full_table_name_mt = TABLE_MASTER # OBS: esto no está amarrado a dp_config
        # Final count for logging
        count_final = spark.table(full_table_name_mt).count()
        logger.info(f"Registros finales en portafolio troncal ({full_table_name_mt}): {count_final:,}")

        total_duration = time.time() - start_time
        logger.log_stage_end('load_preparation_data', duration=total_duration, records_extracted=count_final)
        resultados.append({'mes': mes_actual_proceso, 'registros': count_final, 'exitoso': True, 'duracion': total_duration})

    except Exception as e:
        logger.log_exception(operation='load_preparation_data', exception=e, should_raise=False, mes_vta=mes_actual_proceso)
        resultados.append({'mes': mes_actual_proceso, 'registros': 0, 'exitoso': False, 'error': str(e)})
        continue

In [0]:
logger.info("\n" + "=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")

for r in resultados:
    if r['exitoso']:
        logger.info(f"  ✓ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
    else:
        logger.error(f"  X {r['mes']}: {r.get('error', '?')}")

if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()